# FastAPI - Middleware

**Tematy:**
1. Mechanizm przetwarzania żądań
2. Tworzenie własnych middleware

---
## 1. Czym jest Middleware?

**Middleware = kod wykonywany PRZED i PO każdym requestcie**

In [ ]:
# ═══════════════════════════════════════════════════════════
# Przepływ requestu przez middleware
# ═══════════════════════════════════════════════════════════
"""
Request → [Middleware 1 ↓] → [Middleware 2 ↓] → Endpoint
                                                    ↓
Response ← [Middleware 1 ↑] ← [Middleware 2 ↑] ← Endpoint

Każdy middleware:
1. Dostaje request
2. Może go zmodyfikować / zalogować / zablokować
3. Przekazuje do następnego middleware (lub endpoint)
4. Dostaje response z powrotem
5. Może go zmodyfikować / zalogować
6. Zwraca do poprzedniego middleware

Przykłady użycia:
- Logowanie requestów (timing, headers, body)
- CORS (Cross-Origin Resource Sharing)
- Authentication/Authorization
- Rate limiting
- Custom headers
- Error handling
"""

---
## 2. Najprostszy Middleware

In [ ]:
# ═══════════════════════════════════════════════════════════
# Function-based middleware (@app.middleware)
# ═══════════════════════════════════════════════════════════

from fastapi import FastAPI, Request

app = FastAPI()

@app.middleware("http")
async def simple_middleware(request: Request, call_next):
    """
    Najprostszy middleware
    
    request: Request object
    call_next: funkcja wywołująca następny middleware/endpoint
    """
    # KOD PRZED endpointem
    print(f"Request: {request.method} {request.url.path}")
    
    # Wywołaj endpoint (lub następny middleware)
    response = await call_next(request)
    
    # KOD PO endpoincie (mamy response)
    print(f"Response status: {response.status_code}")
    
    return response

@app.get("/users")
def get_users():
    return [{"id": 1, "name": "John"}]

# Flow:
# 1. Request → middleware: print("Request: GET /users")
# 2. middleware → endpoint: get_users()
# 3. endpoint → middleware: return response
# 4. middleware: print("Response status: 200")
# 5. middleware → client: return response

---
## 3. Przykłady Praktyczne

In [ ]:
# ═══════════════════════════════════════════════════════════
# Timing Middleware (mierzenie czasu requestu)
# ═══════════════════════════════════════════════════════════

import time
from fastapi import FastAPI, Request

app = FastAPI()

@app.middleware("http")
async def timing_middleware(request: Request, call_next):
    # Start timer
    start_time = time.time()
    
    # Wywołaj endpoint
    response = await call_next(request)
    
    # Oblicz czas
    process_time = time.time() - start_time
    
    # Dodaj header do response
    response.headers["X-Process-Time"] = str(process_time)
    
    print(f"{request.method} {request.url.path} - {process_time:.3f}s")
    
    return response

@app.get("/slow")
async def slow_endpoint():
    await asyncio.sleep(1)  # Symulacja wolnego endpointu
    return {"status": "done"}

# GET /slow
# Response headers:
# X-Process-Time: 1.002
# Console:
# GET /slow - 1.002s

In [ ]:
# ═══════════════════════════════════════════════════════════
# Logging Middleware (logowanie requestów i responses)
# ═══════════════════════════════════════════════════════════

import logging
from fastapi import FastAPI, Request

app = FastAPI()

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

@app.middleware("http")
async def logging_middleware(request: Request, call_next):
    # Log request
    logger.info(
        f"Request: {request.method} {request.url.path} "
        f"from {request.client.host}"
    )
    
    # Wywołaj endpoint
    response = await call_next(request)
    
    # Log response
    logger.info(
        f"Response: {response.status_code} for "
        f"{request.method} {request.url.path}"
    )
    
    return response

# Console output:
# INFO: Request: GET /users from 127.0.0.1
# INFO: Response: 200 for GET /users

In [ ]:
# ═══════════════════════════════════════════════════════════
# Custom Headers Middleware
# ═══════════════════════════════════════════════════════════

from fastapi import FastAPI, Request

app = FastAPI()

@app.middleware("http")
async def custom_headers_middleware(request: Request, call_next):
    response = await call_next(request)
    
    # Dodaj custom headers do każdego response
    response.headers["X-Custom-Header"] = "MyApp"
    response.headers["X-Version"] = "1.0.0"
    
    return response

# Każdy response będzie miał:
# X-Custom-Header: MyApp
# X-Version: 1.0.0

In [ ]:
# ═══════════════════════════════════════════════════════════
# Authentication Middleware (sprawdzanie API key)
# ═══════════════════════════════════════════════════════════

from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import JSONResponse

app = FastAPI()

VALID_API_KEYS = {"secret-key-123", "secret-key-456"}

@app.middleware("http")
async def auth_middleware(request: Request, call_next):
    # Pomiń endpointy publiczne
    if request.url.path in ["/", "/health", "/docs"]:
        return await call_next(request)
    
    # Sprawdź API key w headerze
    api_key = request.headers.get("X-API-Key")
    
    if api_key not in VALID_API_KEYS:
        # Zwróć 401 bez wywoływania endpointu
        return JSONResponse(
            status_code=401,
            content={"detail": "Invalid API key"}
        )
    
    # API key OK - wywołaj endpoint
    response = await call_next(request)
    return response

@app.get("/protected")
def protected_endpoint():
    return {"message": "Secret data"}

# GET /protected
# (bez X-API-Key header) → 401 Unauthorized
#
# GET /protected
# X-API-Key: secret-key-123 → 200 OK

---
## 4. Class-based Middleware (BaseHTTPMiddleware)

In [ ]:
# ═══════════════════════════════════════════════════════════
# BaseHTTPMiddleware - class-based approach
# ═══════════════════════════════════════════════════════════

from fastapi import FastAPI, Request
from starlette.middleware.base import BaseHTTPMiddleware
import time

class TimingMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: Request, call_next):
        """
        dispatch() = odpowiednik funkcji middleware
        """
        start_time = time.time()
        
        # Wywołaj endpoint
        response = await call_next(request)
        
        # Oblicz czas
        process_time = time.time() - start_time
        response.headers["X-Process-Time"] = str(process_time)
        
        return response

app = FastAPI()

# Dodaj middleware przez add_middleware()
app.add_middleware(TimingMiddleware)

@app.get("/users")
def get_users():
    return [{"id": 1}]

In [ ]:
# ═══════════════════════════════════════════════════════════
# Class-based Middleware ze stanem
# ═══════════════════════════════════════════════════════════

from fastapi import FastAPI, Request
from starlette.middleware.base import BaseHTTPMiddleware
from collections import defaultdict

class RequestCounterMiddleware(BaseHTTPMiddleware):
    def __init__(self, app):
        super().__init__(app)
        # Stan middleware (licznik requestów per endpoint)
        self.request_counts = defaultdict(int)
    
    async def dispatch(self, request: Request, call_next):
        # Zwiększ licznik
        path = request.url.path
        self.request_counts[path] += 1
        
        print(f"{path} called {self.request_counts[path]} times")
        
        response = await call_next(request)
        return response

app = FastAPI()
app.add_middleware(RequestCounterMiddleware)

@app.get("/users")
def get_users():
    return []

# GET /users → /users called 1 times
# GET /users → /users called 2 times
# GET /users → /users called 3 times

---
## 5. Kolejność Middleware

In [ ]:
# ═══════════════════════════════════════════════════════════
# Kolejność middleware ma znaczenie!
# ═══════════════════════════════════════════════════════════

from fastapi import FastAPI, Request

app = FastAPI()

@app.middleware("http")
async def first_middleware(request: Request, call_next):
    print("1. First middleware - PRZED")
    response = await call_next(request)
    print("6. First middleware - PO")
    return response

@app.middleware("http")
async def second_middleware(request: Request, call_next):
    print("2. Second middleware - PRZED")
    response = await call_next(request)
    print("5. Second middleware - PO")
    return response

@app.middleware("http")
async def third_middleware(request: Request, call_next):
    print("3. Third middleware - PRZED")
    response = await call_next(request)
    print("4. Third middleware - PO")
    return response

@app.get("/test")
def test_endpoint():
    print(">>> ENDPOINT <<<")
    return {"status": "ok"}

# GET /test
# Output:
# 1. First middleware - PRZED
# 2. Second middleware - PRZED
# 3. Third middleware - PRZED
# >>> ENDPOINT <<<
# 4. Third middleware - PO
# 5. Second middleware - PO
# 6. First middleware - PO

# Kolejność:
# - Request: pierwszy zdefiniowany → ostatni zdefiniowany → endpoint
# - Response: endpoint → ostatni zdefiniowany → pierwszy zdefiniowany

In [ ]:
# ═══════════════════════════════════════════════════════════
# Praktyczny przykład kolejności: Auth + Timing
# ═══════════════════════════════════════════════════════════

from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
import time

app = FastAPI()

# 1. Timing - PIERWSZY (zewnętrzny)
@app.middleware("http")
async def timing_middleware(request: Request, call_next):
    start = time.time()
    response = await call_next(request)  # Może wrócić 401 z auth middleware!
    process_time = time.time() - start
    response.headers["X-Process-Time"] = str(process_time)
    return response

# 2. Auth - DRUGI (wewnętrzny)
@app.middleware("http")
async def auth_middleware(request: Request, call_next):
    api_key = request.headers.get("X-API-Key")
    
    if not api_key:
        # Zwróć 401 - timing middleware NADAL zadziała!
        return JSONResponse(
            status_code=401,
            content={"detail": "Missing API key"}
        )
    
    response = await call_next(request)
    return response

# Flow dla niepoprawnego API key:
# 1. timing_middleware (start timer)
# 2. auth_middleware (sprawdź API key)
# 3. auth_middleware (zwróć 401)
# 4. timing_middleware (dodaj X-Process-Time do 401 response)
# 5. Client dostaje 401 z X-Process-Time header

---
## 6. Built-in Middleware (FastAPI + Starlette)

In [ ]:
# ═══════════════════════════════════════════════════════════
# CORSMiddleware (Cross-Origin Resource Sharing)
# ═══════════════════════════════════════════════════════════

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000"],  # Frontend origin
    allow_credentials=True,
    allow_methods=["*"],  # GET, POST, PUT, DELETE, etc.
    allow_headers=["*"],  # Authorization, Content-Type, etc.
)

# Pozwala frontend (localhost:3000) łączyć się z backend (localhost:8000)

In [ ]:
# ═══════════════════════════════════════════════════════════
# TrustedHostMiddleware (security)
# ═══════════════════════════════════════════════════════════

from fastapi import FastAPI
from starlette.middleware.trustedhost import TrustedHostMiddleware

app = FastAPI()

app.add_middleware(
    TrustedHostMiddleware,
    allowed_hosts=["example.com", "*.example.com"]
)

# Blokuje requesty z nieprawidłowym Host header
# Zapobiega Host header injection attacks

In [ ]:
# ═══════════════════════════════════════════════════════════
# GZipMiddleware (kompresja responses)
# ═══════════════════════════════════════════════════════════

from fastapi import FastAPI
from starlette.middleware.gzip import GZipMiddleware

app = FastAPI()

app.add_middleware(GZipMiddleware, minimum_size=1000)  # Kompresuj > 1KB

# Automatycznie kompresuje duże responses (JSON, HTML)
# Oszczędza bandwidth

---
## 7. Zaawansowane - Request Body w Middleware

In [ ]:
# ═══════════════════════════════════════════════════════════
# Czytanie request body w middleware (TRUDNE!)
# ═══════════════════════════════════════════════════════════

from fastapi import FastAPI, Request
import json

app = FastAPI()

@app.middleware("http")
async def log_body_middleware(request: Request, call_next):
    # Problem: request.body() można odczytać tylko RAZ!
    # Po odczytaniu stream jest pusty
    
    # Rozwiązanie: zapisz body i nadpisz request
    body = await request.body()
    
    if body:
        try:
            body_json = json.loads(body)
            print(f"Request body: {body_json}")
        except:
            print(f"Request body (raw): {body[:100]}...")  # Pierwsze 100 bajtów
    
    # WAŻNE: Nie można użyć call_next() z request po odczytaniu body!
    # FastAPI nie może ponownie odczytać body w endpoincie
    
    # Workaround: użyj Request.scope + ASGIRequest
    # (advanced, poza scope tego notebooka)
    
    response = await call_next(request)
    return response

# LEPSZE ROZWIĄZANIE:
# Logowanie body w endpoincie przez dependency (nie middleware)

---
## Podsumowanie

### Middleware w FastAPI

**Czym jest:**
- Kod wykonywany PRZED i PO każdym requestcie
- Request → Middleware → Endpoint → Middleware → Response

**Dwa podejścia:**
1. **Function-based:** `@app.middleware("http")`
2. **Class-based:** `BaseHTTPMiddleware` + `app.add_middleware()`

**Przykłady użycia:**
- ⏱️ Timing (mierzenie czasu requestów)
- 📝 Logging (logowanie requestów/responses)
- 🔒 Authentication (sprawdzanie API key/JWT)
- 📊 Custom headers (dodawanie do każdego response)
- 🌐 CORS (Cross-Origin Resource Sharing)

**Kolejność:**
- Request: pierwszy → ostatni → endpoint
- Response: endpoint → ostatni → pierwszy

**Built-in middleware:**
- `CORSMiddleware` - CORS dla frontend
- `TrustedHostMiddleware` - security (Host header)
- `GZipMiddleware` - kompresja responses

**Ważne:**
- `call_next(request)` wywołuje następny middleware/endpoint
- Middleware może zwrócić response BEZ wywoływania endpointu (auth, rate limiting)
- Request body można odczytać tylko raz (trudne w middleware)

**Kiedy używać middleware:**
✅ Logika dotycząca WSZYSTKICH requestów
✅ Timing, logging, CORS, custom headers
❌ Logika specyficzna dla endpointu (użyj dependencies)